[ 1. Data Collection & Prep ] ──> [ 2. Feature Engineering ] ──> [ 3. Model Training ]
                                                                       │
[ 6. Monitoring & Governance ] <── [ 5. Deployment ] <── [ 4. Evaluation & Validation ]

Craete Model which predicts whether an individual customer is at risk of canceling their subscription (churning) based on their usage patterns and account metrics.

### Data Collection

In [0]:
import pandas as pd
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)
n_samples = 500

# 1. Generate features
customer_ids = np.random.randint(1000, 9999, size=n_samples)
age = np.random.randint(18, 70, size=n_samples)
tenure_months = np.random.randint(1, 72, size=n_samples)
monthly_charges = np.round(np.random.uniform(20.0, 120.0, size=n_samples), 2)
contract_type = np.random.choice(["Month-to-Month", "One-Year", "Two-Year"], size=n_samples, p=[0.5, 0.3, 0.2])
support_tickets = np.random.poisson(lam=1.5, size=n_samples)

# 2. Generate target variable (Churn: 1 = Yes, 0 = No) based on underlying probability rules
# Higher monthly charges and support tickets increase churn risk
churn_prob = 1 / (1 + np.exp(-(-2.0 + 0.03 * monthly_charges + 0.4 * support_tickets - 0.05 * tenure_months)))
churn = np.random.binomial(1, churn_prob)

# 3. Assemble initial DataFrame
df_raw = pd.DataFrame({
    "customer_id": customer_ids,
    "age": age,
    "tenure_months": tenure_months,
    "monthly_charges": monthly_charges,
    "contract_type": contract_type,
    "support_tickets": support_tickets,
    "churn": churn
})

# 4. Intentionally introduce realistic dirty data (missing values and duplicates)
# Inject missing values into 'monthly_charges' and 'age'
df_raw.loc[df_raw.sample(frac=0.04, random_state=42).index, "monthly_charges"] = np.nan
df_raw.loc[df_raw.sample(frac=0.03, random_state=99).index, "age"] = np.nan

# Duplicate the first 10 rows
duplicates = df_raw.iloc[:10].copy()
df_customer = pd.concat([df_raw, duplicates], ignore_index=True)

print(f"Dataset generated successfully! Total rows: {len(df_customer)}")

In [0]:
# Convert Pandas DataFrame to PySpark DataFrame
spark_df = spark.createDataFrame(df_customer)

# Save as a Delta table in Unity Catalog (catalog.schema.table)
# Update catalog/schema names to match your workspace setup if needed
target_table = "workspace.sree_test.raw_customer_churn"

spark_df.write.format("delta").mode("overwrite").saveAsTable(target_table)

print(f"Data saved as Delta table: {target_table}")

In [0]:
%sql
select * from workspace.sree_test.raw_customer_churn

## Feature Engineering

In [0]:
import pandas as pd
import numpy as np

# sklearn modules for preprocessing, building pipelines, and splitting data
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# =====================================================================
# STEP 1: LOAD RAW DATA FROM DELTA LAKE
# =====================================================================
# Read the Unity Catalog Delta table into a PySpark DataFrame, 
# then convert it to a Pandas DataFrame for local scikit-learn processing.
df = spark.table("workspace.sree_test.raw_customer_churn").toPandas()

# =====================================================================
# STEP 2: DATA CLEANING & STRUCTURING
# =====================================================================
# 1. Remove exact duplicate rows to prevent over-counting duplicate records.
df = df.drop_duplicates()

# 2. Drop 'customer_id' column:
# Random ID numbers carry no predictive value. Leaving them in could cause 
# the algorithm to memorize random patterns instead of real trends.
df = df.drop(columns=["customer_id"])

# 3. Separate features (X - Inputs) from target label (y - Output to predict)
X = df.drop(columns=["churn"])  # X contains: age, tenure, monthly_charges, support_tickets, contract_type
y = df["churn"]                 # y contains: 1 (churned) or 0 (stayed)

# =====================================================================
# STEP 3: DEFINE COLUMN TYPES FOR DEDICATED PREPROCESSING
# =====================================================================
# Numeric columns require imputation (filling missing numbers) and scaling.
num_cols = ["age", "tenure_months", "monthly_charges", "support_tickets"]

# Categorical text columns require imputation and binary encoding.
cat_cols = ["contract_type"]

# =====================================================================
# STEP 4: BUILD REUSABLE PREPROCESSING PIPELINES
# =====================================================================

# Pipeline A: How to handle numerical columns step-by-step
num_pipeline = Pipeline(steps=[
    # Step A1: Replace any missing NaN values with the median value of that column
    ("imputer", SimpleImputer(strategy="median")),
    
    # Step A2: Rescale numbers so mean = 0 and std dev = 1
    # Ensures large dollar amounts don't overwhelm small ticket counts
    ("scaler", StandardScaler())
])

# Pipeline B: How to handle text/categorical columns step-by-step
cat_pipeline = Pipeline(steps=[
    # Step B1: Replace missing text with the most frequent category in that column
    ("imputer", SimpleImputer(strategy="most_frequent")),
    
    # Step B2: Convert text categories ("Month-to-Month", "One-Year") into 0/1 binary columns
    # drop="first" drops 1 redundant column to prevent multicollinearity in models
    ("encoder", OneHotEncoder(drop="first", sparse_output=False))
])

# Combine both pipelines into a master preprocessor mapped to specific columns
preprocessor = ColumnTransformer(transformers=[
    ("num_transform", num_pipeline, num_cols),
    ("cat_transform", cat_pipeline, cat_cols)
])

# =====================================================================
# STEP 5: TRAIN / TEST SPLIT (PREVENT DATA LEAKAGE)
# =====================================================================
# Partition data: 80% to train the model, 20% held out to test later.
# stratify=y ensures both splits maintain the exact same ratio of churned vs non-churned customers.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =====================================================================
# STEP 6: EXECUTE TRANSFORMATIONS
# =====================================================================
# Fit the preprocessor ONLY on X_train (learns medians, means, standard deviations of X_train).
# Then apply those exact learned transformation rules to both X_train and X_test.
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Print comparison of shapes before and after transformation
print(f"Original X_train shape: {X_train.shape} (5 columns)")
print(f"Processed X_train shape: {X_train_processed.shape} (6 columns due to One-Hot Encoding)")

##What train_test_split Does?
train_test_split takes your feature dataset (X) and target labels (y) and randomly splits them into two distinct subsets:
Training Set (X_train, y_train): Typically 70%–80% of your data. The machine learning algorithm learns patterns and adjusts its internal weights using this portion.
Testing Set (X_test, y_test): Typically 20%–30% of your data. This data is held out and kept hidden from the model during training. It is used strictly to evaluate how accurately the model generalizes to completely new, unseen data.

Key Parameters to Know
test_size=0.2: Controls the split ratio (0.2 means 20% test data, 80% training data).
random_state=42: Sets a fixed seed for the pseudo-random number generator. Using a fixed number ensures that every time you run the cell, you get the exact same random split (making your ML experiments reproducible).
stratify=y: Ensures that target class distributions are preserved proportionally in both splits (e.g., if 20% of customers in y churned, both y 
train
​	
  and y 
test
​	
  will contain exactly 20% churned examples).

Non-Scikit-Learn Alternatives
While sklearn.model_selection.train_test_split is the gold standard for Python and Pandas,other frameworks provide their own native equivalents for different scale needs:
PySpark (Big Data / Distributed Scale): Uses .randomSplit([0.8, 0.2], seed=42) on PySpark DataFrames.
SQL Data Warehouses: Uses hash-based or row-number sampling (e.g., WHERE ABS(HASH(customer_id)) % 100 < 80).
Deep Learning Frameworks (PyTorch / TensorFlow): Uses torch.utils.data.random_split or tf.keras.utils.

In [0]:
%pip install xgboost

##Batch Inference & Model Monitoring & Governance

In [0]:
import mlflow
import xgboost as xgb
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from mlflow.models import infer_signature

# =====================================================================
# STEP 1: CONFIGURE MLFLOW FOR UNITY CATALOG
# =====================================================================
# Direct MLflow to store registered models in Unity Catalog instead of 
# the legacy workspace-local model registry.
mlflow.set_registry_uri("databricks-uc")

# Enable automatic logging for XGBoost (captures parameters, metrics, feature importances)
mlflow.xgboost.autolog(log_models=False)  # We will manually log the model with explicit signature below

# =====================================================================
# STEP 2: START AN MLFLOW EXPERIMENT RUN
# =====================================================================
# Wrapping training inside 'with mlflow.start_run()' creates a single, 
# trackable experiment run in the Databricks MLflow UI.
with mlflow.start_run(run_name="churn_xgboost_poc_run") as run:
    
    # 1. Instantiate the XGBoost Binary Classifier algorithm with hyperparameters
    model = xgb.XGBClassifier(
        n_estimators=100,        # Number of boosted decision trees to build
        max_depth=4,             # Maximum depth of each decision tree
        learning_rate=0.05,      # Step size shrinkage to prevent overfitting
        random_state=42          # Ensures training results are 100% reproducible
    )
    
    # 2. Train (fit) the model on our Phase 2 processed training features
    model.fit(X_train_processed, y_train)
    
    # 3. Generate predictions on the unseen test set (Phase 2 holdout)
    y_pred = model.predict(X_test_processed)
    y_proba = model.predict_proba(X_test_processed)[:, 1] # Probability scores for Churn = 1
    
    # 4. Calculate classification performance metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    
    # Explicitly log evaluation metrics to MLflow
    mlflow.log_metric("test_accuracy", acc)
    mlflow.log_metric("test_precision", prec)
    mlflow.log_metric("test_recall", rec)
    mlflow.log_metric("test_roc_auc", auc)
    
    print(f"Model Evaluation Metrics on Unseen Test Data:")
    print(f" - Accuracy:  {acc:.4f}")
    print(f" - Precision: {prec:.4f}")
    print(f" - Recall:    {rec:.4f}")
    print(f" - ROC-AUC:   {auc:.4f}\n")
    
    # =====================================================================
    # STEP 3: LOG MODEL WITH UNITY CATALOG SCHEMA SIGNATURE
    # =====================================================================
    # Infer the exact input/output schema so Unity Catalog enforces type safety
    signature = infer_signature(X_train_processed, y_train)
    
    # Log the trained model binary and its preprocessor artifacts to MLflow
    mlflow.xgboost.log_model(
        xgb_model=model,
        artifact_path="model",
        signature=signature
    )
    
    # =====================================================================
    # STEP 4: REGISTER MODEL TO UNITY CATALOG
    # =====================================================================
    # Target model path in Unity Catalog: catalog_name.schema_name.model_name
    uc_model_name = "workspace.sree_test.customer_churn_xgb"
    model_uri = f"runs:/{run.info.run_id}/model"
    
    registered_model = mlflow.register_model(
        model_uri=model_uri,
        name=uc_model_name
    )

print(f"Successfully registered Version {registered_model.version} to Unity Catalog: '{uc_model_name}'")

In [0]:
import mlflow
import pandas as pd
import numpy as np
from pyspark.sql import functions as F

# =====================================================================
# STEP 1: SIMULATE NEW UNSCORED CUSTOMER DATA (DELTA TABLE)
# =====================================================================
# In production, this would be your daily landing Delta table.
np.random.seed(99)
new_customers = pd.DataFrame({
    "customer_id": [2001, 2002, 2003, 2004, 2005],
    "age": [29, 62, 41, 35, 50],
    "tenure_months": [2, 48, 12, 6, 36],
    "monthly_charges": [105.50, 45.00, 89.20, 115.00, 60.00],
    "contract_type": ["Month-to-Month", "Two-Year", "One-Year", "Month-to-Month", "Two-Year"],
    "support_tickets": [5, 0, 2, 4, 1]
})

# Save new batch data to Unity Catalog Delta Lake
unscored_table = "workspace.sree_test.unscored_customers"
spark.createDataFrame(new_customers).write.format("delta").mode("overwrite").saveAsTable(unscored_table)
print(f"Loaded new batch records into '{unscored_table}'")

# =====================================================================
# STEP 2: LOAD NEW DATA & PREPROCESS FEATURES
# =====================================================================
df_new = spark.table(unscored_table).toPandas()
X_new = df_new.drop(columns=["customer_id"])

# Apply the Phase 2 fitted preprocessor (ColumnTransformer)
X_new_processed = preprocessor.transform(X_new)

# =====================================================================
# STEP 3: LOAD REGISTERED MODEL FROM UNITY CATALOG
# =====================================================================
# Set registry URI target to Unity Catalog
mlflow.set_registry_uri("databricks-uc")

# Model URI format: models:/<catalog>.<schema>.<model_name>/<version_or_alias>
uc_model_name = "workspace.sree_test.customer_churn_xgb"
model_version = 1  # Or use an alias like 'models:/main.default.customer_churn_xgb@champion'

model_uri = f"models:/{uc_model_name}/{model_version}"
loaded_model = mlflow.xgboost.load_model(model_uri)
print(f"Successfully loaded '{uc_model_name}' Version {model_version} from Unity Catalog!")

# =====================================================================
# STEP 4: GENERATE PREDICTIONS & PROBABILITIES
# =====================================================================
# Predict binary churn flag (1 = Churn, 0 = Retain)
df_new["predicted_churn"] = loaded_model.predict(X_new_processed)

# Predict churn probability percentage (0.0 to 1.0)
df_new["churn_probability"] = loaded_model.predict_proba(X_new_processed)[:, 1]
df_new["churn_probability"] = df_new["churn_probability"].round(4)

# Assign risk category based on probability threshold
df_new["risk_level"] = np.where(
    df_new["churn_probability"] >= 0.70, "High Risk",
    np.where(df_new["churn_probability"] >= 0.40, "Medium Risk", "Low Risk")
)

# Display predictions
print("\n--- Batch Prediction Results ---")
print(df_new[["customer_id", "monthly_charges", "support_tickets", "predicted_churn", "churn_probability", "risk_level"]])

# =====================================================================
# STEP 5: SAVE PREDICTIONS TO DELTA LAKE (GOVERNED RESULTS)
# =====================================================================
predictions_table = "workspace.sree_test.customer_churn_predictions"

spark.createDataFrame(df_new).write.format("delta").mode("overwrite").saveAsTable(predictions_table)
print(f"\nSaved batch predictions back to Delta Lake: '{predictions_table}'")

## Test Service Endpoint
{
  "inputs": [
    [-0.51, -0.98, 1.15, 5.0, 0.0, 0.0]
  ]
}

##Testing Endpoint

In [0]:
import requests
import json
import pandas as pd

# Fetch workspace URL & auth token
workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

endpoint_name = "customer-churn-service"  # Replace with your endpoint name
url = f"https://dbc-1c21bcdb-1dfb.cloud.databricks.com/serving-endpoints/PredictCustomerRisk/invocations"

# 1. Raw customer input
raw_sample = pd.DataFrame([{
    "age": 29,
    "tenure_months": 2,
    "monthly_charges": 105.50,
    "contract_type": "Month-to-Month",
    "support_tickets": 5
}])

# 2. Transform into the 6 preprocessed features expected by the model
processed_matrix = preprocessor.transform(raw_sample)

# 3. Format as 'inputs' tensor payload
payload = {
    "inputs": processed_matrix.tolist()
}

headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}

response = requests.post(url, headers=headers, data=json.dumps(payload))
print("Status Code:", response.status_code)
print("Prediction Response:", response.json())

###1. Unity Catalog Model Registry vs. Workspace Model Registry

Question: What is the structural difference between registering a model in Unity Catalog versus the legacy Workspace Model Registry, and how does it impact enterprise data governance?

Answer: Workspace registries are isolated to a single Databricks workspace and governed via workspace-level ACLs. Unity Catalog introduces a 3-level namespace (catalog.schema.model) with centralized ANSI SQL access control across multiple workspaces. Crucially, Unity Catalog captures end-to-end lineage from source Delta tables and notebooks down to the registered model version and served endpoint.
###2. Handling Model Artifacts & MLflow Logging Protocols

Question: In an enterprise ML pipeline, why should you avoid logging raw model objects directly, and how do you leverage MLflow to ensure reproducible serving endpoints?

Answer: Logging just the raw model binary omits essential feature preprocessing logic, leading to schema mismatches when serving endpoints receive raw payloads. Using mlflow.sklearn.log_model() or custom mlflow.pyfunc models allows you to encapsulate both the preprocessing pipeline (ColumnTransformer) and the estimator, alongside environment dependencies (conda.yaml or requirements.txt) and input signatures (infer_signature).

###3. Model Lifecycle & Zero-Downtime Deployment via Aliases

Question: How do Unity Catalog model aliases (@Champion, @Challenger) improve CI/CD and deployment stability compared to traditional numerical model versions?

Answer: Hardcoding version numbers (e.g., Version 1) in serving endpoints or downstream batch jobs requires code updates and manual re-deployments whenever a new model is trained. Aliases act as mutable pointers. You can point your serving endpoint to models:/catalog.schema.model@Champion. When a new model (Version 2) passes validation, you assign @Champion to Version 2 via MLflow API or UI, updating the endpoint with zero downtime and no code changes.

###4. Signature Enforcement & Schema Validation in Model Serving

Question: What causes a 400 BAD_REQUEST: Failed to enforce schema error when invoking a Databricks Model Serving endpoint, and how does MLflow enforce input validation?

Answer: During mlflow.log_model(), MLflow infers or accepts an explicit schema signature via infer_signature(X_train, y). Databricks Model Serving uses this metadata at the REST API gateway layer to validate incoming JSON structures (dataframe_records, dataframe_split, or inputs). If data types (e.g., passing a string where float64 is expected) or matrix dimensions mismatch, the endpoint rejects the request before it reaches the model execution container.

###5. Production Monitoring with Inference Tables

Question: How does Databricks capture endpoint invocation payloads for MLOps monitoring and data drift detection without impacting endpoint latency?

Answer: By enabling Inference Tables on the serving endpoint. Databricks asynchronously logs all incoming REST request payloads, model predictions, latency metrics, and execution IDs directly into a governed Delta Lake table in Unity Catalog. Because this logging occurs out-of-band via a background stream, it captures real-world data for drift detection and model performance monitoring without adding synchronous overhead or latency to client responses.